# Signer's Doubt: Phases 3-4: baseline audit, SafeTune hardening, guard

### Setup
Runtime → Change runtime type → **T4 GPU** (or better). Accept the Tiny Aya licence at https://huggingface.co/CohereLabs/tiny-aya-global first: the weights are gated.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'NO GPU: switch the runtime to a GPU'

In [ ]:
import os
if not os.path.exists('signers-doubt'):
    !git clone -q https://github.com/davieslennox0/signers-doubt.git
%cd signers-doubt
!pip uninstall -y -q torchao            # SafeTune's Colab note
!pip install -q -r requirements.txt 2>&1 | tail -3
!pip check | head -20

In [ ]:
from huggingface_hub import login, whoami
login()        # token with read access to Tiny Aya and write access to your namespace
HF_USER = whoami()['name']; print(HF_USER)

### Baseline
`base` and `base_boundary` on `test` and `heldout_format`, plus a sampled run. Traces resume if the session drops.

In [ ]:
!python -m eval.run_all --only base base_boundary base_sampled

**Checkpoint:** if baseline ASR is under 15%, stop here: the attacks are too weak or the agent too passive, and training would show nothing.

In [ ]:
print(open('reports/results_test.md').read()); print(open('reports/results_heldout_format.md').read())

### Capability before hardening (AuditKIT: MMLU and GSM8K slices)

In [ ]:
!python -m eval.capability --out reports/capability_base

### Guard data needs the base agent's real mistakes on the *train* split

In [ ]:
!python -m eval.run_all --only base --splits train --runs runs/train --reports runs/train_reports
!python -m training.build_sft
!python -m training.guard_data --traces runs/train/base_train.jsonl

### 4a. Agent hardening with SafeTune SafeGrad, and the plain-SFT ablation

In [ ]:
!python -m training.harden_safetune --method safegrad --out outputs/hardened-safegrad --epochs 2

In [ ]:
!python -m training.harden_safetune --method plain --out outputs/hardened-plain --epochs 2

### 4b. Guard

In [ ]:
!python -m training.harden_safetune --method plain --data data/sft/guard_train.jsonl --val data/sft/guard_val.jsonl --out outputs/guard --epochs 2

In [ ]:
from IPython.display import Image, display
for d in ('hardened-safegrad', 'hardened-plain', 'guard'): display(Image(f'outputs/{d}/training_curve.png'))

### Quick test-set numbers before the full evaluation

In [ ]:
!python -m eval.run_all --only base hardened hardened_guard --splits test --limit 60 --runs runs/quick --reports runs/quick_reports --no-sampled --agent-adapter outputs/hardened-safegrad/adapter --guard-adapter outputs/guard/adapter

In [ ]:
!python scripts/publish_hf.py --user {HF_USER} --what models